<p align="center">
  <span style="color:Navy; font-size:200%; font-weight:bold; vertical-align:middle;">
        Estadística y Probabilidad    
  </span>
</p>
<p align="center" style="line-height:1.2;">
  <span style="color:RoyalBlue; font-size:160%;">Tema 4: Distribuciones muestrales y pruebas de hipótesis </span><br/>
  <span style="color:DodgerBlue; font-size:140%;"> Pruebas de hipótesis </span><br/>
  <span style="font-size:100%;color:forestgreen"> Escuela Nacional de Ciencias de la Tierra  |  Semestre 2027-I</span>
</p>

---

# **<font color="Navy">  Laboratorio 6: Pruebas de hipótesis </font>**


---
# 1. Introducción teórica

## 4.5 Prueba de hipótesis
Una **hipótesis estadística** es una afirmación sobre un parámetro poblacional (media $\mu$, mediana $\tilde\mu$, proporción, varianza...) o sobre la forma de una distribución. Una **prueba de hipótesis** es un procedimiento que usa una muestra para decidir si los datos aportan evidencia suficiente en contra de una afirmación.

**Componentes de una prueba**

| Componente | Qué es | Ejemplo |
|---|---|---|
| Hipótesis nula $H_0$ | Afirmación "de partida", siempre con igualdad | $\mu = 2.65$ g/cm³ |
| Hipótesis alternativa $H_1$ | Lo que queremos respaldar con evidencia | $\mu \neq 2.65$, $\mu > 2.65$ o $\mu < 2.65$ |
| Nivel de significancia $\alpha$ | Probabilidad máxima tolerada de error tipo I | 0.05 |
| Estadístico de prueba | Resume los datos y tiene distribución conocida bajo $H_0$ | $t=\dfrac{\bar x-\mu_0}{s/\sqrt n}$ |
| Región crítica (de rechazo) | Valores del estadístico que llevan a rechazar $H_0$ | $t > t_{crit}$ |
| Valor p | Probabilidad, **suponiendo $H_0$ cierta**, de un resultado al menos tan extremo como el observado | $p = 0.012$ |
| Decisión | Rechazar $H_0$ si $p<\alpha$ (o si el estadístico cae en la región crítica) | Rechazar |

**Errores**

| | $H_0$ verdadera | $H_0$ falsa |
|---|---|---|
| Rechazo $H_0$ | **Error tipo I** (prob. $\alpha$) | Decisión correcta (**potencia** $=1-\beta$) |
| No rechazo $H_0$ | Decisión correcta | **Error tipo II** (prob. $\beta$) |

**Flujo de trabajo:** (1) plantear $H_0$ y $H_1$ → (2) fijar $\alpha$ → (3) elegir la prueba y verificar supuestos → (4) calcular estadístico y valor p → (5) decidir → (6) concluir en el lenguaje del problema.

> ⚠️ *No rechazar $H_0$* **no** es *demostrar* $H_0$: solo significa que no hay evidencia suficiente en su contra (puede haber poca potencia).

## 4.6 Pruebas paramétricas vs. no paramétricas

* **Paramétricas:** suponen un modelo para la distribución (típicamente normal) y contrastan parámetros como la media. Si los supuestos se cumplen, son las más potentes. Ejemplo: $t$ de una muestra.
* **No paramétricas:** hacen pocos supuestos sobre la distribución; suelen trabajar con signos o rangos y contrastan la mediana o la ubicación de la distribución. Son más robustas a atípicos y asimetría, con una pequeña pérdida de potencia cuando los datos sí son normales. Ejemplos: prueba del signo, Wilcoxon.

En ciencias de la Tierra muchas variables (concentraciones, caudales, precipitación diaria, magnitudes sísmicas, conductividad hidráulica) son **asimétricas con colas largas**; ahí la mediana y los métodos no paramétricos (o una transformación logarítmica) suelen ser más adecuados.

## 4.7 Pruebas sobre la media y la mediana

| Prueba | Contrasta | Supuestos principales | En `scipy` |
|---|---|---|---|
| $t$ de una muestra | Media $\mu$ | Independencia; datos aprox. normales (o $n$ grande) | `stats.ttest_1samp(x, mu0, alternative=...)` |
| Signo | Mediana $\tilde\mu$ | Solo independencia | `stats.binomtest(k, n, 0.5, alternative=...)` |
| Wilcoxon de rangos con signo | Mediana (centro de simetría) | Independencia y **simetría** | `stats.wilcoxon(x - m0, alternative=...)` |
| Bootstrap para la mediana | Mediana | Independencia; remuestreo | Se programa con `numpy` |

El argumento `alternative` toma los valores `"two-sided"`, `"greater"` (cola derecha) o `"less"` (cola izquierda).

---
# 2. Datos de los ejemplos
Usaremos datos de:
* `densidad`: densidad (g/cm³) de 32 muestras de granito. Valor de referencia de la literatura: 2.65 g/cm³.
* `nitratos`: nitratos (mg/L) en 35 pozos. Distribución asimétrica. Límite de referencia: 10 mg/L.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# ---- Datos para la clase (no modificar) ----

rng = np.random.default_rng(7)
densidad = rng.normal(2.70, 0.06, 32).round(3)
nitratos = rng.lognormal(mean=np.log(8), sigma=0.6, size=35).round(1)

pd.DataFrame({
    "variable": ["densidad", "nitratos"],
    "n": [len(densidad), len(nitratos)],
    "media": [densidad.mean(), nitratos.mean()],
    "mediana": [np.median(densidad), np.median(nitratos)],
    "desv_est": [densidad.std(ddof=1), nitratos.std(ddof=1)],
}).round(3)

---
# 3. Ejemplo A · Formular hipótesis y componentes de la prueba

### A.1 Formular hipótesis
Pregunta: *¿la densidad media del granito en la zona es mayor que la de referencia (2.65 g/cm³)?*

* $H_0: \mu = 2.65$
* $H_1: \mu > 2.65$ → prueba **unilateral derecha**

**Truco:** $H_1$ es lo que el investigador quiere respaldar; la dirección de la desigualdad sale de la redacción de la pregunta ("mayor que", "menor que", "difiere de").

### A.2 Calcular los componentes paso a paso

In [ ]:
mu0 = 2.65
alpha = 0.05

n = len(densidad)
xbar = densidad.mean()
s = densidad.std(ddof=1)
se = s / np.sqrt(n)                       # error estándar

t_obs = (xbar - mu0) / se                 # estadístico de prueba
gl = n - 1                                # grados de libertad
t_crit = stats.t.ppf(1 - alpha, gl)       # valor crítico (cola derecha)
p_valor = stats.t.sf(t_obs, gl)           # P(T >= t_obs) bajo H0

print(f"n = {n} | media = {xbar:.4f} | s = {s:.4f} | EE = {se:.4f}")
print(f"t_obs = {t_obs:.3f} | gl = {gl} | t_crit = {t_crit:.3f}")
print(f"valor p = {p_valor:.4f}")
print("Criterio del valor crítico:", "Rechazar H0" if t_obs > t_crit else "No rechazar H0")
print("Criterio del valor p      :", "Rechazar H0" if p_valor < alpha else "No rechazar H0")

# Verificación con scipy
print(stats.ttest_1samp(densidad, mu0, alternative="greater"))

**Interpretación:** si el valor p es menor que $\alpha$, los datos son poco compatibles con $H_0$ y se rechaza. Se escribe la conclusión en el lenguaje del problema, por ejemplo: *"Con un nivel de significancia de 5 %, hay evidencia de que la densidad media del granito en la zona es mayor que 2.65 g/cm³."*

### A.3 Visualizar la región crítica

In [ ]:
x = np.linspace(-4, 4, 500)
y = stats.t.pdf(x, gl)

fig, ax = plt.subplots()
ax.plot(x, y, color="k")
ax.fill_between(x[x >= t_crit], y[x >= t_crit], color="tab:red", alpha=0.4,
                label=f"Región crítica (α = {alpha})")
ax.axvline(t_obs, color="tab:blue", ls="--", label=f"t obs = {t_obs:.2f}")
ax.set_xlabel("t"); ax.set_ylabel("Densidad")
ax.set_title("Prueba t unilateral derecha")
ax.legend()
plt.show()

**Adaptación a otras alternativas:**
* Bilateral: la región crítica son **dos colas**, $|t|>t_{1-\alpha/2}$; el valor p es `2 * stats.t.sf(abs(t_obs), gl)`.
* Unilateral izquierda: $t < t_{\alpha}$, con `stats.t.ppf(alpha, gl)`; el valor p es `stats.t.cdf(t_obs, gl)`.

---
# 4. Ejemplo B · Error tipo I y potencia por simulación

En la vida real usualmente tenemos una sola una muestra, así que no puedemos saber con qué frecuencia se equivoca una prueba. En una simulación, podemos fijar la verdad (el valor real de la media), generamos miles de muestras y contamos cuántas veces la prueba rechaza H₀. Así se ve el comportamiento de la prueba a largo plazo, que es lo que significan α y la potencia.
**Error tipo I**

Qué se está probando

El error tipo I es rechazar una H₀ que es verdadera. Para medirlo hay que garantizar que H₀ sea verdadera. Por eso se simula con la media real igual a la hipotetizada:



### B.1 Error tipo I
Simulamos muchas muestras con $H_0$ **verdadera** ($\mu=2.65$, $\sigma=0.06$). La proporción de rechazos debe aproximarse a $\alpha$.

Técnica: generar una matriz de `reps × n` y aplicar la prueba por filas con `axis=1`.

In [ ]:
reps, n_sim = 5000, 32
muestras = rng.normal(2.65, 0.06, size=(reps, n_sim))
pvals = stats.ttest_1samp(muestras, 2.65, axis=1).pvalue

for a in [0.10, 0.05, 0.01]:
    print(f"alpha = {a:.2f} -> proporción de rechazos = {(pvals < a).mean():.4f}")

Cuando H₀ es verdadera, el valor p tiene distribución uniforme entre 0 y 1: es tan probable obtener p < 0.05 como p entre 0.50 y 0.55. Entonces:

P(p < 0.10) = 10 %

P(p < 0.05) = 5 %

P(p < 0.01) = 1 %

Eso es exactamente lo que significa “nivel de significancia α”: es la tasa de falsas alarmas que aceptas de antemano. Si usas α = 0.05, aceptas que, en el 5 % de los casos en que H₀ es cierta, concluirás erróneamente que no lo es.

### B.2 Potencia
Ahora la media real es **2.68** (H₀ es falsa). La proporción de rechazos es la **potencia** $1-\beta$. La potencia crece con $n$.


mu_real es la verdad (de donde salen los datos) y mu0 es lo que afirma H₀ (lo que se contrasta). La distancia entre ambos, 0.03 g/cm³, es el efecto que la prueba debería detectar.

Se usa alternative="greater" porque el efecto va hacia arriba y H₁ es μ > 2.65.

La proporción de rechazos ahora es una cosa distinta: es la proporción de veces que la prueba acierta al detectar el efecto. Eso es la potencia (1 − β).

In [ ]:
def potencia(n, mu_real=2.68, mu0=2.65, sigma=0.06, alpha=0.05, reps=5000):
    muestras = rng.normal(mu_real, sigma, size=(reps, n))
    p = stats.ttest_1samp(muestras, mu0, axis=1, alternative="greater").pvalue
    return (p < alpha).mean()

tamanos = [5, 10, 20, 30, 50, 80]
pot = [potencia(n) for n in tamanos]

plt.plot(tamanos, pot, "o-")
plt.axhline(0.8, color="r", ls="--", label="Potencia = 0.80")
plt.xlabel("Tamaño de muestra n"); plt.ylabel("Potencia (1 - β)")
plt.legend(); plt.show()
print(dict(zip(tamanos, np.round(pot, 3))))

**Lectura:** más datos, más potencia; efectos más grandes o menos variabilidad, también. Con $\beta = 1 -$ potencia, una potencia de 0.80 significa 20 % de probabilidad de no detectar un efecto real de ese tamaño.

---
# 5. Ejemplo C · Revisar supuestos y robustez

### C.1 Diagnóstico de normalidad
Combina **gráficos** (histograma, Q-Q) con **números** (asimetría, Shapiro-Wilk). Con $n$ pequeño Shapiro tiene poca potencia, así que no lo uses solo.

In [ ]:
datos = {"densidad": densidad, "nitratos": nitratos}

fig, axs = plt.subplots(2, 2, figsize=(9, 7))
filas = []
for j, (nombre, d) in enumerate(datos.items()):
    axs[0, j].hist(d, bins=10, edgecolor="k")
    axs[0, j].set_title(nombre)
    stats.probplot(d, dist="norm", plot=axs[1, j])
    axs[1, j].set_title("")
    sw = stats.shapiro(d)
    filas.append({"variable": nombre, "asimetria": stats.skew(d),
                  "W": sw.statistic, "p_shapiro": sw.pvalue})
plt.tight_layout(); plt.show()

In [ ]:
pd.DataFrame(filas).round(4)

**Cómo leerlo**
* Q-Q: los puntos sobre la recta indican normalidad; una curva indica asimetría.
* Shapiro-Wilk: $H_0$ = los datos provienen de una normal. Un valor p pequeño sugiere que **no** son normales.
* Decisión práctica: `densidad` → prueba $t$ sobre la media; `nitratos` (asimétrica) → pruebas sobre la mediana, o $t$ sobre `np.log(nitratos)`.

### C.2 Sensibilidad a valores atípicos
Agregamos un pozo con 200 mg/L de nitratos y comparamos los estadísticos.

In [ ]:
nitratos_out = np.append(nitratos, 200)

for nombre, d in [("sin atípico", nitratos), ("con atípico", nitratos_out)]:
    t = stats.ttest_1samp(d, 10)
    print(f"{nombre:12s} media={d.mean():7.2f}  mediana={np.median(d):6.2f}  "
          f"s={d.std(ddof=1):7.2f}  p(t, bilateral)={t.pvalue:.4f}")

La media y la desviación estándar se disparan; la mediana casi no se mueve. Por eso, ante datos con atípicos conviene reportar y contrastar la **mediana**.

---
# 6. Ejemplo D · Pruebas sobre la media
*(Base para los ejercicios 3.1 y 3.2)*

### D.1 Prueba bilateral con intervalo de confianza
Pregunta: *¿la densidad media difiere de 2.65 g/cm³?* → $H_0:\mu=2.65$ vs. $H_1:\mu\neq2.65$.

In [ ]:
res = stats.ttest_1samp(densidad, 2.65)                 # bilateral por defecto
ic = res.confidence_interval(confidence_level=0.95)

print(res)
print(f"IC 95% para la media: ({ic.low:.4f}, {ic.high:.4f})")
print("¿2.65 está dentro del IC?", ic.low <= 2.65 <= ic.high)

**Equivalencia útil:** en una prueba bilateral al nivel $\alpha$, se rechaza $H_0:\mu=\mu_0$ si y solo si $\mu_0$ queda **fuera** del intervalo de confianza de nivel $1-\alpha$.

### D.2 Unilateral vs. bilateral

In [ ]:
uni = stats.ttest_1samp(densidad, 2.65, alternative="greater")
bi = stats.ttest_1samp(densidad, 2.65, alternative="two-sided")

print("Unilateral derecha:", round(uni.pvalue, 5))
print("Bilateral         :", round(bi.pvalue, 5))
print("Cociente bilateral / unilateral =", round(bi.pvalue / uni.pvalue, 2))

Cuando el estadístico va en la dirección de $H_1$, el valor p bilateral es el **doble** del unilateral. La alternativa se elige **antes** de mirar los datos.

---
# 7. Ejemplo E · Pruebas sobre la mediana

Pregunta: *¿la mediana de nitratos es menor que el límite de 10 mg/L?* → $H_0:\tilde\mu=10$ vs. $H_1:\tilde\mu<10$.

### E.1 Prueba del signo, Wilcoxon y $t$
* **Signo:** cuenta cuántos datos superan el valor de referencia. Bajo $H_0$, esa cuenta sigue una Binomial$(n, 0.5)$. Se descartan los datos iguales al valor de referencia.
* **Wilcoxon:** usa los **rangos** de las diferencias; supone simetría alrededor de la mediana.

In [ ]:
m0 = 10
dif = nitratos - m0
n_eff = int((dif != 0).sum())
k = int((dif > 0).sum())           # cuántos exceden el valor de referencia

signo = stats.binomtest(k, n_eff, 0.5, alternative="less")
wil = stats.wilcoxon(dif, alternative="less")
tt = stats.ttest_1samp(nitratos, m0, alternative="less")

print(f"Pozos > {m0} mg/L: {k} de {n_eff}")
print(f"Prueba del signo   : p = {signo.pvalue:.4f}")
print(f"Wilcoxon           : p = {wil.pvalue:.4f}")
print(f"t (sobre la media) : p = {tt.pvalue:.4f}")

**Comparación:** la prueba del signo usa menos información (solo el signo), por lo que suele ser la menos potente; Wilcoxon aprovecha la magnitud pero exige simetría; la $t$ exige normalidad. Con datos asimétricos, una opción es transformar con logaritmo y contrastar contra $\ln(10)$:

In [ ]:
t_log = stats.ttest_1samp(np.log(nitratos), np.log(m0), alternative="less")
print(f"t sobre log(nitratos) contra log(10): p = {t_log.pvalue:.4f}")

---
# 8. Plantilla de decisión: ¿qué prueba uso?

1. ¿Qué parámetro quiero contrastar: media o mediana?
2. Grafico los datos (histograma, Q-Q) y reviso asimetría y atípicos.
3. Elijo:

| Situación | Prueba sugerida |
|---|---|
| Aprox. normal, sin atípicos serios (o $n$ grande) | $t$ de una muestra sobre la media |
| Asimétrica pero simétrica tras `log` | $t$ sobre `log(x)` o Wilcoxon sobre `log(x)` |
| Simétrica con atípicos o colas pesadas | Wilcoxon sobre la mediana |
| Asimétrica, sin querer suponer nada | Prueba del signo o bootstrap para la mediana |

4. Defino $H_0$, $H_1$ y $\alpha$ **antes** de calcular.
5. Calculo el estadístico y el valor p, y decido.
6. Concluyo en el lenguaje del problema, y reporto el **intervalo de confianza o tamaño del efecto**, no solo "significativo / no significativo".

**Errores comunes a evitar**
* Decir "se acepta $H_0$" (lo correcto: "no se rechaza $H_0$").
* Interpretar el valor p como la probabilidad de que $H_0$ sea cierta.
* Elegir la alternativa unilateral después de ver los datos.
* Usar la $t$ con datos muy asimétricos y $n$ pequeño sin revisar supuestos.
* Confundir significancia estadística con relevancia práctica.

# Ejercicios:

In [ ]:
# -------- Datos para los ejercicios ----------------------

rng = np.random.default_rng(2026)

# pH de 30 muestras de suelo agrícola
ph_suelo = rng.normal(6.2, 0.45, 30).round(2)

# Arsénico en agua de 40 pozos (µg/L): distribución asimétrica
arsenico = rng.lognormal(mean=np.log(12), sigma=0.7, size=40).round(1)

# Temperatura media anual (°C) en una estación meteorológica, 36 años
temp_anual = rng.normal(16.2, 2.1, 36).round(1)

# Caudal medio diario (m³/s) de un río en 25 días de estiaje-lluvia: asimétrico
caudal = rng.lognormal(mean=np.log(35), sigma=0.9, size=25).round(1)

pd.DataFrame({
    "variable": ["ph_suelo", "arsenico", "temp_anual", "caudal"],
    "n": [len(ph_suelo), len(arsenico), len(temp_anual), len(caudal)],
    "media": [ph_suelo.mean(), arsenico.mean(), temp_anual.mean(), caudal.mean()],
    "mediana": [np.median(ph_suelo), np.median(arsenico), np.median(temp_anual), np.median(caudal)],
}).round(2)

---
## Ejercicio 1.1 · Formular hipótesis estadísticas
Para cada situación escribe $H_0$ y $H_1$ usando símbolos ($\mu$ para la media, $\tilde{\mu}$ para la mediana) e indica si la prueba es **bilateral**, **unilateral derecha** o **unilateral izquierda**.

| # | Situación |
|---|---|
| a | El pH medio de un suelo agrícola difiere del valor óptimo de 6.5. |
| b | La mediana de arsénico en pozos excede el límite de 10 µg/L. |
| c | La temperatura media anual aumentó respecto a los 15.0 °C del periodo 1961–1990. |
| d | La mediana del caudal del río es menor que 60 m³/s. |

**Tu respuesta:**

| # | $H_0$ | $H_1$ | Tipo de prueba |
|---|---|---|---|
| a | | | |
| b | | | |
| c | | | |
| d | | | |

## Ejercicio 1.2 · Componentes de una prueba, paso a paso
Contrasta $H_0: \mu = 15.0$ vs. $H_1: \mu > 15.0$ con los datos `temp_anual`, **sin usar** `ttest_1samp` al principio. Calcula:

1. $n$, $\bar{x}$, $s$ (con `ddof=1`) y el error estándar $s/\sqrt{n}$.
2. El estadístico $t_{obs} = \dfrac{\bar{x}-\mu_0}{s/\sqrt{n}}$.
3. Los grados de libertad y el **valor crítico** $t_{crit}$ para $\alpha=0.05$ (`stats.t.ppf`).
4. El **valor p** (`stats.t.sf` o `1 - stats.t.cdf`).
5. La **decisión** (rechazar o no $H_0$) por ambos criterios: valor crítico y valor p.
6. Verifica tu resultado con `stats.ttest_1samp(..., alternative="greater")`.

Responde:
1. ¿Qué representa el valor p en este contexto? Redáctalo con tus palabras, sin decir "probabilidad de que $H_0$ sea cierta".
2. Escribe la conclusión en lenguaje de climatología (no estadístico).

## Ejercicio 1.3 · Visualizar la región crítica
Grafica la densidad $t$ con los grados de libertad del ejercicio anterior. Sombrea la **región de rechazo** (cola derecha, $\alpha = 0.05$) y marca con una línea vertical el $t_{obs}$.

---
# Pruebas paramétricas vs. no paramétricas (4.6)

## Ejercicio 2.1 · Cuadro conceptual
Completa el cuadro.

| | Paramétrica (ej. $t$ de una muestra) | No paramétrica (ej. Wilcoxon, signo) |
|---|---|---|
| Parámetro que contrasta | | |
| Supuestos sobre la distribución | | |
| Usa los valores originales o los rangos/signos | | |
| Sensibilidad a valores atípicos | | |
| Potencia si se cumplen los supuestos | | |
| Variables típicas en ciencias de la Tierra donde conviene | | |

## Ejercicio 2.2 · Revisar supuestos
Para `ph_suelo`, `arsenico` y `caudal`:

1. Dibuja un histograma y un gráfico Q-Q (`stats.probplot`) en una figura de 2 filas × 3 columnas.
2. Calcula la asimetría (`stats.skew`) y la prueba de Shapiro-Wilk (`stats.shapiro`).
3. Resume en una tabla de `pandas`.

¿Para cuáles variables te parece razonable usar una prueba paramétrica sobre la media? ¿Por qué?

---
# Pruebas sobre la media y la mediana (4.7)

## Ejercicio 3.1 · Prueba $t$ bilateral para la media
Un agrónomo afirma que el pH óptimo es 6.5. Con `ph_suelo`:

1. Plantea $H_0$ y $H_1$.
2. Aplica `stats.ttest_1samp` (bilateral).
3. Calcula el **intervalo de confianza del 95 %** para $\mu$ (usa `stats.t.interval` o `res.confidence_interval()`).
4. Concluye, y explica la relación entre el IC y la decisión de la prueba.

## Ejercicio 3.2 · Prueba $t$ unilateral
Con `temp_anual` contrasta $H_0:\mu=15.0$ vs. $H_1:\mu>15.0$ usando `alternative="greater"`. Luego repítela con $H_1:\mu \neq 15$ y compara los valores p.

¿Cuál es la relación entre el valor p unilateral y el bilateral cuando el estadístico va en la dirección de $H_1$?

## Ejercicio 3.3 · Prueba del signo y de Wilcoxon sobre la mediana
Contrasta $H_0:\tilde{\mu}=10$ vs. $H_1:\tilde{\mu}>10$ µg/L con `arsenico`.

1. **Prueba del signo:** cuenta cuántos pozos exceden 10 µg/L (`k`) y usa `stats.binomtest(k, n, 0.5, alternative="greater")`, donde `n` es el número de datos distintos de 10.
2. **Wilcoxon de rangos con signo:** `stats.wilcoxon(arsenico - 10, alternative="greater")`.
3. Compara con la prueba $t$ sobre la media (`alternative="greater"`).

¿Qué supuesto adicional exige Wilcoxon que no exige la prueba del signo? ¿Cuál de las tres pruebas te parece más defendible para estos datos y por qué?